In [ ]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, precision_recall_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import joblib

# ==============================
# 2. LOAD DATASET
# ==============================
df = pd.read_csv("loan_approval_dataset_cleaned.csv")
print(df.head())

# ==============================
# 3. DATA PREPROCESSING
# ==============================

# Handle missing values
df = df.ffill()

# Encode categorical variables
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

# Split features and target
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ==============================
# 4. TRAIN MODELS
# ==============================
lr = LogisticRegression()
dt = DecisionTreeClassifier()
rf = RandomForestClassifier()
svm = SVC(probability=True)

lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
svm.fit(X_train, y_train)

# ==============================
# 5. MODEL EVALUATION (ACCURACY)
# ==============================
models = {
    "Logistic Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "SVM": svm
}

results = {}

print("\n=== Model Accuracy ===")
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name}: {acc:.4f}")

# Select best model automatically
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

print(f"\nBest Model: {best_model_name}")

# ==============================
# 6. CONFUSION MATRIX
# ==============================
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

disp.plot()
plt.title(f"Confusion Matrix ({best_model_name})")
plt.show()

# ==============================
# 7. ROC CURVE + AUC
# ==============================
y_prob = best_model.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# ==============================
# 8. PRECISION-RECALL CURVE
# ==============================
precision, recall, _ = precision_recall_curve(y_test, y_prob)

plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

# ==============================
# 9. FEATURE IMPORTANCE (RF ONLY)
# ==============================
if best_model_name == "Random Forest":
    importance = best_model.feature_importances_
    features = X.columns

    feat_imp = pd.Series(importance, index=features).sort_values()

    feat_imp.plot(kind='barh')
    plt.title("Feature Importance")
    plt.show()

# ==============================
# 10. TYPE I & TYPE II ERRORS
# ==============================
tn, fp, fn, tp = cm.ravel()

print("\n=== Error Analysis ===")
print("Type I Error (False Positive - Risky loan approved):", fp)
print("Type II Error (False Negative - Good customer rejected):", fn)

# ==============================
# 11. SAVE MODEL & SCALER
# ==============================
joblib.dump(best_model, "loan_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("\nModel and scaler saved successfully!")

# ==============================
# 12. SAVE RESULTS TABLE
# ==============================
results_df = pd.DataFrame(list(results.items()), columns=["Model", "Accuracy"])
print("\n=== Model Comparison Table ===")
print(results_df)
